<a href="https://colab.research.google.com/github/lyn1126/bigdata-nhom6/blob/main/notebooks/01_kiem_tra_du_lieu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 – Kiểm tra dữ liệu ban đầu
**Vai trò:** Kiểm tra nhanh cấu trúc 9 bảng của Olist, tỷ lệ giao trễ sơ bộ và trạng thái các đơn thiếu ngày giao.


In [1]:
!pip install -q kagglehub

import pandas as pd
import kagglehub

DATA = kagglehub.dataset_download("olistbr/brazilian-ecommerce") + '/'

customers = pd.read_csv(DATA + 'olist_customers_dataset.csv')
orders    = pd.read_csv(DATA + 'olist_orders_dataset.csv')
items     = pd.read_csv(DATA + 'olist_order_items_dataset.csv')
payments  = pd.read_csv(DATA + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(DATA + 'olist_order_reviews_dataset.csv')
products  = pd.read_csv(DATA + 'olist_products_dataset.csv')
sellers   = pd.read_csv(DATA + 'olist_sellers_dataset.csv')
geo       = pd.read_csv(DATA + 'olist_geolocation_dataset.csv')

for name, df in [('customers',customers),('orders',orders),('items',items),
                 ('payments',payments),('reviews',reviews),('products',products),
                 ('sellers',sellers),('geolocation',geo)]:
    print(f'{name:12s} {df.shape}')

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
customers    (99441, 5)
orders       (99441, 8)
items        (112650, 7)
payments     (103886, 5)
reviews      (99224, 7)
products     (32951, 9)
sellers      (3095, 4)
geolocation  (1000163, 5)


In [2]:
print(orders.columns.tolist())
print(customers.columns.tolist())
orders.head()

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [3]:
# Chuyển các cột ngày sang kiểu datetime
for c in ['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']:
    orders[c] = pd.to_datetime(orders[c])

# (1) Tỷ lệ giao trễ, chỉ tính các đơn đã có ngày giao thực tế
delivered = orders.dropna(subset=['order_delivered_customer_date']).copy()
delivered['is_late'] = (delivered['order_delivered_customer_date']
                        > delivered['order_estimated_delivery_date']).astype(int)
print('Đơn có ngày giao thực tế:', len(delivered))
print('Tỷ lệ giao trễ: %.2f%%' % (delivered['is_late'].mean()*100))

# (2) Tỷ lệ khách mua từ 2 đơn trở lên (dùng customer_unique_id)
oc = orders.merge(customers[['customer_id','customer_unique_id']], on='customer_id')
n_orders = oc.groupby('customer_unique_id')['order_id'].nunique()
print('Số khách duy nhất:', len(n_orders))
print('Tỷ lệ khách mua >= 2 đơn: %.2f%%' % ((n_orders >= 2).mean()*100))

# (3) Tỷ lệ đơn bị hủy
print('Phân bố trạng thái đơn:')
print(orders['order_status'].value_counts(normalize=True).mul(100).round(2))

# (4) Số dòng thiếu ngày giao thực tế
print('Thiếu ngày giao thực tế:', orders['order_delivered_customer_date'].isna().sum())

Đơn có ngày giao thực tế: 96476
Tỷ lệ giao trễ: 8.11%
Số khách duy nhất: 96096
Tỷ lệ khách mua >= 2 đơn: 3.12%
Phân bố trạng thái đơn:
order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64
Thiếu ngày giao thực tế: 2965


In [4]:
# Trạng thái của các đơn thiếu ngày giao thực tế
missing = orders[orders['order_delivered_customer_date'].isna()]
print('Trạng thái của đơn thiếu ngày giao:')
print(missing['order_status'].value_counts())

# Đơn "delivered" nhưng thiếu ngày giao
print('\nDelivered nhưng thiếu ngày giao:',
      ((orders['order_status']=='delivered') & orders['order_delivered_customer_date'].isna()).sum())

# Kiểm tra trùng khóa ở các bảng sẽ nhúng/tham chiếu
print('\nReview: số order_id duy nhất / tổng dòng:',
      reviews['order_id'].nunique(), '/', len(reviews))
print('Đơn có >1 món hàng:', (items.groupby('order_id').size() > 1).sum())
print('Đơn có >1 lần thanh toán:', (payments.groupby('order_id').size() > 1).sum())

# Geolocation: số mã zip duy nhất
print('\nGeolocation: zip prefix duy nhất:', geo['geolocation_zip_code_prefix'].nunique())

Trạng thái của đơn thiếu ngày giao:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Delivered nhưng thiếu ngày giao: 8

Review: số order_id duy nhất / tổng dòng: 98673 / 99224
Đơn có >1 món hàng: 9803
Đơn có >1 lần thanh toán: 2961

Geolocation: zip prefix duy nhất: 19015
